# Question 1: Text Preprocessing and Representation for Social Media Data

This notebook answers the whole of Question 1. It explains text preprocessing for noisy social media text, demonstrates cleaning and tokenization on the given tweet, and shows Bag-of-Words and TF-IDF representations.

## (a) Text preprocessing and suitable techniques

### (a) i. Concept and importance

Text preprocessing is the process of converting raw text into a cleaner, more consistent, and machine-readable form before applying NLP algorithms. Social media data is usually noisy because it contains mixed capitalization, spelling variations, URLs, hashtags, mentions, emojis, slang, repeated punctuation, and informal grammar.

For disease outbreak monitoring, preprocessing is important because it helps a public health organization detect useful signals from public posts. For example, `COVID`, `covid`, `Covid!!!`, and `#COVID` may all refer to the same concept. Normalizing such text improves later tasks such as classification, sentiment analysis, trend detection, and alert generation.

### (a) ii. Two suitable preprocessing techniques

| Technique | How it works | Impact on informal social media text |
|---|---|---|
| URL, mention, and hashtag handling | Removes or normalizes links and mentions; hashtags can be kept as words. | Reduces noise while preserving important topic words such as `StaySafe`. |
| Normalization of case, punctuation, and repeated characters | Converts text to lowercase, removes excessive punctuation, and can reduce repeated letters. | Makes words consistent, reduces vocabulary size, and improves matching between similar posts. |

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## (b) Tokenization and cleaning of the given tweet

Given tweet:

> OMG!!! COVID cases rising. Visit https://news.com NOW!!! #StaySafe

In [ ]:
tweet = "OMG!!! COVID cases rising. Visit https://news.com NOW!!! #StaySafe"
tweet

In [ ]:
def clean_social_text(text):
    """Clean noisy social media text for simple NLP representation."""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    return re.findall(r"\b\w+\b", text)


stop_words = {"the", "is", "a", "an", "and", "or", "to", "of", "in", "on", "for", "was", "now"}

cleaned_text = clean_social_text(tweet)
tokens = tokenize(cleaned_text)
tokens_without_stopwords = [token for token in tokens if token not in stop_words]

cleaned_text, tokens, tokens_without_stopwords

In [ ]:
steps = pd.DataFrame(
    [
        (1, "Original text", tweet),
        (2, "Lowercasing", tweet.lower()),
        (3, "URL removal", re.sub(r"https?://\S+|www\.\S+", "", tweet.lower()).strip()),
        (4, "Remove punctuation", re.sub(r"[^a-z0-9#\s]", " ", re.sub(r"https?://\S+|www\.\S+", "", tweet.lower())).strip()),
        (5, "Hashtag handling", cleaned_text),
        (6, "Tokenization", tokens),
        (7, "Optional stop-word removal", tokens_without_stopwords),
    ],
    columns=["Step", "Operation", "Result"],
)

steps

### (b) ii. Final cleaned representation and explanation

Final cleaned tokens:

`['omg', 'covid', 'cases', 'rising', 'visit', 'staysafe']`

The text is converted to lowercase so that `COVID` and `covid` are treated as the same word. The URL is removed because it adds little meaning for this classification example. Repeated punctuation is removed because it increases noise. The hashtag is kept as the word `staysafe` because it carries health-related meaning. The word `now` is removed as a stop word in this compact representation, although it could be retained if the task were urgency detection.

## (c) Bag-of-Words and TF-IDF representation

### (c) i. Bag-of-Words

A Bag-of-Words representation counts word occurrences and ignores grammar and word order. Using the cleaned tokens, every term appears once.

In [ ]:
final_text = " ".join(tokens_without_stopwords)

bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform([final_text])

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out(),
)

bow_df

In [ ]:
term_frequencies = bow_df.T.reset_index()
term_frequencies.columns = ["Term", "Frequency"]
term_frequencies

Vector form using the vocabulary `['cases', 'covid', 'omg', 'rising', 'staysafe', 'visit']` is:

`[1, 1, 1, 1, 1, 1]`

### (c) ii. How TF-IDF adjusts term importance

Bag-of-Words gives each word occurrence equal weight. TF-IDF, meaning Term Frequency-Inverse Document Frequency, adjusts a word's weight by considering how often it appears in one document and how rare it is across a collection of documents. Common terms receive lower scores, while more specific terms receive higher scores.

For example, words such as `visit` may be less informative if they appear in many posts, while terms such as `covid`, `cases`, and `staysafe` may be more useful for outbreak monitoring.

In [ ]:
sample_posts = [
    final_text,
    "covid cases reported today",
    "visit clinic for covid testing",
    "football match starts today",
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(sample_posts)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
)

tfidf_df.round(3)

## (d) Influence and limitation of preprocessing

### (d) i. Influence on downstream NLP tasks

- It improves classification accuracy by helping models learn useful patterns instead of noise.
- It reduces vocabulary size by merging forms such as `COVID`, `covid`, and `covid!!!`.
- It improves feature quality by removing URLs and excess punctuation.
- It supports better generalization because new social media posts are represented more consistently.

### (d) ii. One limitation and its effect

A limitation of preprocessing is that it can remove useful meaning. For example, emojis may show fear, sadness, anger, or relief, and repeated punctuation may show urgency or strong emotion. Removing the word `now` may also reduce the ability to detect urgency. Therefore, preprocessing choices should depend on the NLP task instead of being applied blindly.

## Final answer summary

Question 1 shows that social media text must be cleaned and normalized before NLP analysis. The given tweet becomes the compact representation `['omg', 'covid', 'cases', 'rising', 'visit', 'staysafe']`. Bag-of-Words represents these words using counts, while TF-IDF improves the representation by giving more importance to terms that are more distinctive across a dataset. Good preprocessing improves downstream NLP tasks, but excessive cleaning can remove important sentiment or urgency signals.